# Session 1 — LLM Anatomy
### What it is, and what it is not

---

You are the AI lead at **Velora**, a European e-bike retailer. You are considering 3 AI Providers: (Google, Z.ai, DeepSeek). 

By the end of this notebook you will have your own numbers to help you make an informed decision.

You will rank the three models by eye, and then spend the rest of the session finding out whether that ranking survives contact with a measurement:
- tokens across six languages 
- a 3 × 3 × 5 grid of latency 
- cost 
- a machine-computed quality score

five identical calls at `temperature=0`, and three prompts every model gets confidently wrong.

Today the **prompt is fixed and only the model varies**. 

Shaping the request: (roles, formats, constraints is next week). 
Vary two things at once and you cannot attribute the difference to either.

| | | |
|---|---|---|
| 0. Setup | 10 min | one key, the data, your first call |
| A. Ranking by eye | 15 min | your ranking, and the room's |
| B. The tokenizer | 20 min | your language multiplier |
| C. The instrumented grid | 25 min | p50/p95, cost, quality score |
| D. Determinism | 15 min | 5 identical calls, count distinct |
| E. The edge cases | 15 min | hallucination rate, 3 × 3 |
| F. Bonus | 10 min | free |

**Deliverable:** this executed notebook (end of session) plus a one-page report
(due next Thursday, 11pm ).

## 0. Setup

You have been given **one** key. It reaches all three models you are about to compare:
one built by Google, one by Z.AI (GLM), one by DeepSeek.

If your key does not work, raise your hand.


| model | who built it | reached as |
|---|---|---|
| **Gemini** | Google | `google/gemini-3.5-flash-lite` |
| **GLM** | Z.ai | `z-ai/glm-5.3-flash` |
| **DeepSeek** | open weights | `deepseek/deepseek-chat-v3.1` |

In [ ]:
!pip install -q openai # Install the OpenAI Python client library

import json, os, time, urllib.request, collections
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass
from openai import OpenAI
import numpy as np
import pandas as pd

print("ready")

In [ ]:
# Everything this lab needs is in one public folder.
BASE = "https://raw.githubusercontent.com/vorpax/velora-labs/refs/heads/main/s1/"

FILES   = ["s1_prompts.json", "s1_texts.json", "s1_product_facts.json"]

for name in FILES:
    urllib.request.urlretrieve(BASE + name, name) # We just download the files to the current working directory (directory: fancy name for folder).

print("downloaded:", sorted(p for p in os.listdir() if not p.startswith(".")))

In [ ]:
PROMPTS = json.load(open("s1_prompts.json"))        # 9 prompts, 3 families
TEXTS   = json.load(open("s1_texts.json"))          # the same description, 6 languages
FACTS   = json.load(open("s1_product_facts.json"))  # product sheet + 8 extractable facts

BY_PID   = {p["id"]: p for p in PROMPTS}
FAMILIES = sorted({p["family"] for p in PROMPTS})

print(f"{len(PROMPTS)} prompts in {len(FAMILIES)} families: {', '.join(FAMILIES)}")
print(f"{len(TEXTS)} language versions of one product description")
print(f"{len(FACTS['facts'])} extractable facts on the product sheet")
print()
print(PROMPTS[3]["prompt"]) # print one of the prompts to see what it looks like

In [ ]:
# The three models, and what each one costs. Prices are EUR per 1M tokens.

MODELS = {
    "gemini": dict(
        model="google/gemini-3.5-flash-lite",
        price_in=0.10, price_out=0.40,
        direct_base_url="https://openrouter.ai/api/v1",
    ),
    "zai": dict(
        model="z-ai/glm-5.3-flash",
        price_in=0.20, price_out=0.60,
        direct_base_url="https://openrouter.ai/api/v1",
    ),
    "deepseek": dict(
        model="deepseek/deepseek-chat-v3.1",
        price_in=0.28, price_out=0.88,
        direct_base_url="https://openrouter.ai/api/v1",
    ),
}
print(f"{len(MODELS)} models: " + ", ".join(f"{k} ({v['model']})" for k, v in MODELS.items()))

In [ ]:
# One key, one client, three models. max_retries makes the client itself re-send a
# request that failed on a transport error.
key = getpass("OpenRouter API key: ")

client = OpenAI(api_key=key.strip(), base_url="https://openrouter.ai/api/v1",
                max_retries=5)

print("endpoint:", client.base_url)

### The call

The next cell is the entire API surface this session uses. 
Everything later in the notebook is this call in a loop.

Notice that all three models go through the *same* client code. 
Reaching Gemini from Google directly instead of through OpenRouter is two changes:
- the `direct_base_url` in the `MODELS` dict
- An API key from Google

Lock-in is a choice, not a fact.

In [ ]:
print("prompt:", BY_PID["P1"]["prompt"][:500]) # What does the prompt look like?

In [ ]:
# THE CALL. Every measurement in this notebook comes from this shape.

r = client.chat.completions.create(
    model=MODELS["gemini"]["model"],
    messages=[{"role": "user", "content": BY_PID["P1"]["prompt"]}],
    temperature=0,
)

print(r.choices[0].message.content[:500])
print()
print("input tokens :", r.usage.prompt_tokens)
print("output tokens:", r.usage.completion_tokens)

In [ ]:
# The same four lines, plus the accounting, so the grid can call it 135 times.
# Instrumenting a call before having an opinion about it is the habit of this session.

LOG = []          # We record every call there, in order to compare.

def call_model(name, prompt, rep=1, prompt_id=None, temperature=0.0):
    cfg = MODELS[name] # We get the model configuration
    t0 = time.time() # t0 : current time (measured in seconds, for latency)
    
    r = client.chat.completions.create(model=cfg["model"], temperature=temperature,
                                       messages=[{"role": "user", "content": prompt}]) # We make a call to the model, with the prompt and temperature.
    
    t1 = time.time() # t1 : current time (measured in seconds, for latency)
    
    token_input = r.usage.prompt_tokens # How many tokens were in the prompt
    token_output =  r.usage.completion_tokens # How many tokens were in the output
    
    request_cost = token_input * cfg["price_in"] / 1e6 + token_output * cfg["price_out"] / 1e6
    
    m = dict(
             model=name, # name of the model
             prompt_id=prompt_id, # id of the prompt
             rep=rep, # Number of repetitions
             latency_s=round(t1 - t0, 3), # latency in seconds (rounded to 3 decimal places)
             in_tokens=token_input, 
             out_tokens=token_output, 
             cost_eur= request_cost, 
             temperature=temperature
             )
    
    LOG.append(m) # We add the metrics to the log, so we can analyze them later.
    return r.choices[0].message.content or "", m # We return the text and the metrics.

print("call_model() is ready. It will return the text and the metrics of the call, and log the metrics in LOG.")


In [ ]:
print("-"*50, "prompt for P4 (first 200 characters):")
print(BY_PID["P4"]["prompt"][:200])



_text, _metrics = call_model("zai", BY_PID["P4"]["prompt"], prompt_id="P4")

print("-"*50, "text returned by the model (first 120 characters):")
print(_text[:120]) # a fancy way to print the first 120 characters of the text returned by the model
print(_metrics)

## A. Ranking by eye · 15 min

The normal way a company picks a model: run the same prompt through the candidates,
read the outputs, pick the one that feels better. Do exactly that, carefully — then
keep the result, because the rest of the session is spent testing it.

Do not confer with the next table, and do not adjust a score after hearing someone
else's. The exercise only works if your numbers are honestly yours.

The next cell sends **one** generative prompt to the three models and prints the three
answers, unlabelled.

In [ ]:
EYE_PROMPT = "P8"      # the warranty reply: two clauses that disagree
eye_outputs = {}
for name in MODELS: # We call each model with the same prompt, and store the output in a dictionary.
    txt, _ = call_model(name, BY_PID[EYE_PROMPT]["prompt"], rep=1, prompt_id=EYE_PROMPT) # We do call the model with the wrapper we've previously defined.
    eye_outputs[name] = txt # Then save it into a dict.

print("PROMPT:\n" + BY_PID[EYE_PROMPT]["prompt"]) # What the prompt looks like. It is the same for all models.

for i, m in enumerate(MODELS):
    print("\n" + "=" * 74)
    print(f"OUTPUT {chr(65+i)}")
    print("=" * 74)
    print(eye_outputs[m])

In [ ]:
# =============================== TODO 1 ======================================
# Score each of the three outputs from 1 (unusable) to 5 (ship it).
#
# BEFORE you scroll back up and read them again: write, in one sentence, what a 5
# means. 
# 
# Not plainly "good/ a good answer/ a correct answer". 
# A rule you could hand to a colleague and get the same number back. 
# 
# That sentence is the actual deliverable of this TODO; the three integers are
# just what it produces.
#
# This matters more than it looks. In Part C you will need a quality score that a
# machine can compute. 
# Whatever you cannot state precisely enough for a colleague to reproduce, you cannot state precisely enough to automate either.

WHAT_A_5_MEANS = ...     # <- one sentence, as a string (wrap it in ""). Write it first.
WHAT_A_5_MEANS_BAD_EXAMPLE = "A 5 means the answer is good." 

MY_SCORES = {
    "A": ...,            # <- 1 to 5
    "B": ...,
    "C": ...,
}

assert isinstance(WHAT_A_5_MEANS, str) and len(WHAT_A_5_MEANS) > 30, \
    "Write the rule before the numbers. This is the point of the exercise."
assert all(isinstance(v, int) and 1 <= v <= 5 for v in MY_SCORES.values())

print(WHAT_A_5_MEANS)
print(MY_SCORES, "-> your top pick:", max(MY_SCORES, key=MY_SCORES.get))

### The room, in twenty seconds

Hands up for A, then for B, then for C. The instructor calls out the three counts and
you type them in.

In [ ]:
# Top picks only, from the show of hands. Three integers, called out by the instructor.
ROOM_TOP_PICKS = {"A": ..., "B": ..., "C": ...}

assert all(isinstance(v, int) for v in ROOM_TOP_PICKS.values()), \
    "Fill in the three counts from the show of hands before running this."

n = sum(ROOM_TOP_PICKS.values())
bloc, bloc_n = max(ROOM_TOP_PICKS.items(), key=lambda kv: kv[1])
mine = max(MY_SCORES, key=MY_SCORES.get)

print(f"pairs voting     : {n}")
print(f"largest bloc     : {bloc}  ({bloc_n}/{n} = {bloc_n / n:.0%})")
print(f"pairs outside it : {n - bloc_n}/{n}")
print(f"your top pick    : {mine}" + ("  (with the bloc)" if mine == bloc else "  (against it)"))

### What this tells you

You just ranked three models by reading one answer from each. So did everyone
else in the room. Whether the room agreed or not, that is the whole evidence
base: one sample per model, read once.

This is how most companies pick a model. It is also what a vendor demo is,
except that the vendor chose the sample.

The problem is not your judgement, it is the method. Reading one answer per model
is a rough measurement. 

If two models are far apart, it will show. 
If they are close, one sample cannot tell them apart, no matter how carefully you read it.
You do not know yet which case you are in.

Keep `MY_SCORES`. In Part C you will rank the same three models again, this time
with a score computed by code over 45 calls, and compare the two rankings. 

The question is not which ranking is correct, but how much they differ.

*For your report:* your top pick, the room's split, and one sentence on what
would have had to be true for you to trust the eye ranking.

## B. The tokenizer · 20 min

The model does not read your text. It reads a sequence of integers produced by a
tokenizer — a compression scheme fitted on a training corpus, chosen because it makes
training efficient, not because it carves language at any meaningful joint.

`s1_texts.json` holds **one** Velora product description in six languages. Same facts,
same structure, six translations. Anything that differs between them differs because of
the tokenizer and nothing else.

In [ ]:
print("-"*50, "text for each language", "-"*50, "\n")

for t in TEXTS:
    print("-"*50, f"prompt for {t['label']} (first 200 characters):")
    print(t["text"][:200])
    print('\n')

In [ ]:
# We measure tokens with a real API call rather than a local tokenizer library: 
# In fact the number that appears on your invoice is the provider's count.
# Moreover, tokenizers might be different across models.

# One call per language on one model

TOK_MODEL = "gemini" 

rows = [] 
for t in TEXTS:
    _, m = call_model(TOK_MODEL, t["text"], rep=1, prompt_id=f"TOK-{t['lang']}") # We call the LLM.
    
    rows.append({"lang": t["lang"],  # language code, e.g. "en" for English
                 "label": t["label"], # the name of the language in English
                 "script": t["script"], # the alphabet or writing system used in the language
                 "chars": len(t["text"]), # number of characters
                 "words": len(t["text"].split()), # number of words
                 "in_tokens": m["in_tokens"]}) # input_tokens



In [ ]:
tok = pd.DataFrame(rows).set_index("lang") # We create a DataFrame from the rows, and set the index to the language code.
tok["tokens_per_char"] = tok.in_tokens / tok.chars # We compute the number of tokens per character for each language.
tok["vs_english"] = tok.in_tokens / tok.loc["en", "in_tokens"] # We compute the ratio of tokens for each language compared to English.

In [ ]:
display(tok[["label", "script", "chars", "in_tokens", "tokens_per_char", "vs_english"]]
        .style.format({"tokens_per_char": "{:.3f}", "vs_english": "{:.2f}x"})) # display the DataFrame

In [ ]:

worst = tok.vs_english.idxmax() # We find the language with the highest ratio of tokens compared to English.
print(f"\nYOUR MULTIPLIER: {tok.vs_english.max():.2f}x  ({worst} vs en)  <- for your report") 

**What the multiplier costs, and what it actually costs.**

Velora support handles ~2 000 emails a week. 

At ~320 input and ~120 output tokens each, 0.30 €/M in and 2.50 €/M out, that is about **41 €/year**:
**10 € of input** and **31 € of output** — the bill is three quarters output.

Your Polish or Greek multiplier applies to the **input side only** — the output does not move.
At ~2.2×: `10 × 2.2 + 31` ≈ **53 €/year**, a differential of about twelve euros.
Nothing changes except the language.

Twelve euros a year is a rounding error, and saying so is the point: the first-order
effect is small. The second-order one is not.

And the effect that does not scale away: **the same context window holds ~2.2× fewer emails.** A window is measured in tokens, so a multiplier on the input is a divisor on how much fits. 

You meet that again in session 3, when a corpus has to fit in a prompt, and in session 4, when an execution trace does.

### TODO 2 — two decision rules, written before you have the numbers

You have to recommend **one** model for two different Velora products, using the
measurements you are about to take in Part C.

| | volume | who reads the output |
|---|---|---|
| **(a) internal support triage** | ~400 requests/day | staff-facing; a human reviews every output before it reaches a customer |
| **(b) customer-facing chat** | ~8 000 requests/day | the answer *is* the product; the customer sees it unedited |

Write both rules **now**, before you have the numbers.

Each one must name the metric it keys on and the threshold that flips the decision.
If the same rule wins both, you have written a preference, not a decision rule — the two
cases differ in what a mistake costs and in who absorbs it, and a rule that ignores that
is not a rule.


In [ ]:
# =============================== TODO 2 ======================================

DECISION_RULE_TRIAGE = ...   # <- a string (Wrap it in "")
DECISION_RULE_CHAT   = ...   # <- a string

assert all(isinstance(x, str) and len(x) > 60 for x in (DECISION_RULE_TRIAGE, DECISION_RULE_CHAT))
assert DECISION_RULE_TRIAGE != DECISION_RULE_CHAT, "Same rule for both? Re-read the two cases."
print("triage:", DECISION_RULE_TRIAGE)
print("chat  :", DECISION_RULE_CHAT)

## C. The instrumented grid · 25 min

Three models × three prompt families × five repetitions. Everything is recorded:
latency, tokens, cost, and a quality score that a machine computes.

**The quality score is deterministic.** On the extraction family it checks whether each
of the eight Velora product facts appears in the output, using the accepted surface 
forms in `s1_product_facts.json`: 

For instance : `720` and `720 Wh` pass, `750` does not. 


No model grades another model, we will do LLM as a Judge in Session 5



In [ ]:
# For now, we don't care about output formatting. For instance : 24.6kg and 24.6kg are the same thing.

def normalise(s):
    """Lowercase, collapse whitespace, neutralise separators that are pure formatting.

    Non-breaking spaces and comma decimals are locale, not content: a scorer that marks
    `24,6` wrong when it wanted `24.6` is measuring where the model thinks it lives.
    """
    s = str(s).lower().replace("\u00a0", " ").replace("\u202f", " ") # Remove non-breaking spaces.
    return " ".join(s.replace(",", ".").split()) # Replace commas with dots, split by whitespace, and join back with single spaces.

print(normalise("Weight\u00a0: 24,6 kg")) # a test to see if the normalise function works correctly.

In [ ]:
# ------------------------------------------------- the deterministic scorer
def score_extraction(text):
    """How many of the 8 product facts appear, matched PER LINE against their key.

    We are asking for a `key: value` shape.
    Hence, a fact only counts when its key and one of its accepted forms land on the
    same line.
    In session 2, we will see a more robust way to format extraction (Structured output).
    
    We ignore: fluency, ordering, whether the answer arrived wrapped in an apology or after an "You are ABSOLUTLY RIGHT :)" 
    """
    
    lines = [normalise(l) for l in str(text).splitlines() if l.strip()] # Split the text into lines, normalise each line, and ignore empty lines.
    hits = []
    
    for f in FACTS["facts"]:
        key = normalise(f["key"]) # We normalise the key of the fact.
        for l in lines: # We iterate over each line in the text.
            if key not in l: # If the key is not in the line, we skip to the next line.
                continue 
            val = l.split(":", 1)[1] if ":" in l else l # We extract the value part of the line.
            if any(f" {normalise(a)} " in f" {val} " for a in f["accepted"]): # Check if any of the accepted forms of the fact are present in the value part of the line.
                hits.append(f["key"])
                break
    return len(hits), hits

def score_abstention(text):
    """T-UNANSWERABLE: did it say it does not know, instead of inventing?"""
    t = normalise(text)
    return any(k in t for k in ("not_stated", "not stated", "do not have",
                                "don\'t have", "cannot verify", "no information",
                                "not specified", "unknown"))



In [ ]:
_demo = "battery_capacity_wh: 720 Wh\nweight_kg_size_m: 24,6\nprice_eur: 3 290" # A test string that contains three facts in the expected format.
print("scorer smoke test:", score_extraction(_demo), "(expect 3 facts)")  # A test to see if the score_extraction function works correctly.

In [ ]:
N_REPS = 5

In [ ]:
# =============================== TODO 3 ======================================
# The grid below runs each (model, prompt) pair 5 times. That number was chosen
# to fit the session, not because it is right.
#
# Write one sentence saying what those 5 repetitions measure. "To be more
# reliable" does not count. Reliable about what: the median latency, the slowest
# calls, how much the quality score moves, how often a call fails? Each of those
# needs a different number of repetitions.
#
# Then write the number you would actually need for the thing you named.
#
# Worth knowing: with 5 repetitions, the "p95 latency" printed later is just the
# slowest of your 5 calls. That is not a p95. It is better to use a weak number
# and know it is weak than to trust one you cannot explain.


N_REPS_IDEAL = ...            # <- an integer
WHY_N_REPS = ...        # <- one sentence naming what the repetitions estimate

assert isinstance(N_REPS_IDEAL, int) and 1 <= N_REPS_IDEAL <= 10_000
assert isinstance(WHY_N_REPS, str) and len(WHY_N_REPS) > 40
print(f"the grid runs N_REPS = {N_REPS}; you would need {N_REPS_IDEAL}: {WHY_N_REPS}")

In [ ]:
# ------------------------------------------------------------ worker function to parallelize the work
def process(task): # runs one (model, prompt, rep) combination
    name, p, rep = task
    txt, m = call_model(name, p["prompt"], rep=rep, prompt_id=p["id"]) # call the model with the wrapper

    n_facts, _ = score_extraction(txt) if p["family"] == "extraction" else (None, []) # score extraction if relevant

    return {
        **m, # unpack the metrics dictionary into the new dictionary
        "family": p["family"], # add the family of the prompt
        "trap": p["trap"], # add the potential trap of the prompt
        "text": txt,
        "facts_found": n_facts, # add the number of facts found
        "abstained": score_abstention(txt), # add whether the model abstained from answering
    }

# ------------------------------------------------------------------ the grid
# 3 models x 9 prompts x N_REPS (5).
tasks = [(name, p, rep) for name in MODELS for p in PROMPTS for rep in range(1, N_REPS + 1)] # build the full task list

rows = [] # a list to hold the results
with ThreadPoolExecutor(max_workers=100) as ex: # Slowed by the network, not the CPU.
    futures = {ex.submit(process, t): t for t in tasks} # submit all tasks, keep a map back to the original params
    for f in as_completed(futures): # collect results as they finish (unordered)
        try:
            rows.append(f.result())
            print(f"Task completed for {futures[f]}") # log successful calls
        except Exception as e:
            print(f"Task failed for {futures[f]}: {e}") # log and skip failed calls instead of crashing the whole grid

In [ ]:
grid = pd.DataFrame(rows)
print(f"{len(grid)} calls  ({len(MODELS)} models x {len(PROMPTS)} prompts x {N_REPS} reps)")
grid.head(3)

In [ ]:
# ---------------------------------------------------- the table for your report
summary = grid.groupby("model").agg(
    calls        = ("model", "size"),
    p50_latency  = ("latency_s", "median"),
    p95_latency  = ("latency_s", lambda s: s.quantile(0.95)),
    in_tokens    = ("in_tokens", "mean"),
    out_tokens   = ("out_tokens", "mean"),
    eur_per_call = ("cost_eur", "mean"),
)
summary["eur_per_1000"] = summary.eur_per_call * 1000

qual = (grid[grid.family == "extraction"]
        .groupby("model").facts_found.agg(["mean", "min", "max"])
        .rename(columns={"mean": "facts_mean", "min": "facts_min", "max": "facts_max"}))
summary = summary.join(qual)

display(summary.style.format({
    "p50_latency": "{:.2f}s", "p95_latency": "{:.2f}s",
    "in_tokens": "{:.0f}", "out_tokens": "{:.0f}",
    "eur_per_call": "{:.6f}", "eur_per_1000": "{:.3f}",
    "facts_mean": "{:.1f}/8", "facts_min": "{:.0f}", "facts_max": "{:.0f}"}))

print("\nFOR YOUR REPORT: p95 latency and EUR per 1000 calls, per model, from the table above.")
print("And the caveat that belongs next to them: with", N_REPS,
      "repetitions the p95 is the maximum of", N_REPS, "observations.")

In [ ]:
# ------------------------------------------- your eye ranking, against your grid
# A, B and C were printed in MODELS order, so the mapping was fixed all along.
LABEL = dict(zip("ABC", MODELS))

by_eye      = [LABEL[o] for o in sorted(MY_SCORES, key=MY_SCORES.get, reverse=True)]
by_measure  = summary.facts_mean.sort_values(ascending=False).index.tolist()

print("by eye     :", " > ".join(by_eye), "   (1 generative answer, read by you)")
print("by measure :", " > ".join(by_measure), f"   ({N_REPS * 3} extraction calls, scored by code)")
print("same winner:", by_eye[0] == by_measure[0])
print()
print("These two rankings do not answer the same question, and that is the point: one")
print("model can write the better warranty reply and extract the fewer facts. 'Which")
print("model is better' is not a well-formed question until you have said better at what.")

### The number that is not in that table

Every column above is a cost per *call*. None is a cost per **task correctly
accomplished**, which is the only one a business decision can rest on

You cannot compute it today, because it needs a definition of "correctly" on cases you
did not pick after seeing the outputs. That is session 5. 

P3 asks for the same eight facts as P1, wrapped in ~380 tokens of market context that
change nothing about the answer. Compare the two input token counts: **the bill is
driven by what you send, not by what you asked for.**

In [ ]:
print("\n" + "=" * 30, "P1 Prompt", "=" * 30)
print(BY_PID["P1"]["prompt"][:500]) # print the first 500 characters of the prompt for P1

print("\n" + "=" * 30, "P3 Prompt", "=" * 30)
print(BY_PID["P3"]["prompt"][:500]) # print the first 500 characters of the prompt for P1


In [ ]:
# P1 vs P3 — same eight facts requested, very different invoice.
t4 = (grid[grid.prompt_id.isin(["P1", "P3"])]
      .groupby(["model", "prompt_id"])
      .agg(in_tokens=("in_tokens", "mean"), facts=("facts_found", "mean"),
           eur=("cost_eur", "mean")).round(6))

display(t4)
print("Same answer requested. The ratio of the two input token counts is the ratio of "
      "the two bills, and no model 'chose' it — you did, when you wrote the prompt.")

## D. Determinism does not exist · 15 min

`temperature=0` means: at every step, take the most likely token. 

That sounds like a guarantee of reproducibility bit It is not one. 

Both server-side batching and mixture-of-experts can send the same input down a different
path. 
On a more mathematical level floating point addition might change what the most likely token is.

Five identical calls. Count the distinct answers.

In [ ]:
DET_PROMPT = "P1"      # extraction: the most constrained prompt in the grid
det_rows = []
for name in MODELS:
    outs = [call_model(name, BY_PID[DET_PROMPT]["prompt"], rep=r, prompt_id=DET_PROMPT,
                       temperature=0.0)[0] for r in range(1, 6)]
    distinct = len({normalise(o) for o in outs})
    facts = {score_extraction(o)[0] for o in outs}
    det_rows.append({"model": name, "calls": len(outs), "distinct_outputs": distinct,
                     "distinct_fact_counts": len(facts), "fact_counts": sorted(facts)})

det = pd.DataFrame(det_rows).set_index("model")
display(det)
print("\nFOR YOUR REPORT: distinct outputs out of 5, per model.")
print("Then the question that matters: did the SCORE change, or only the wording?")

**Read the two columns separately, because they say different things.**

`distinct_outputs` counts how many byte-distinct strings came back. 

`distinct_fact_counts`counts how many different *scores* those strings earned. 

For instance, if the first is 3 and the second is 1, the model wandered in wording and never in substance — which is a system you can
build on.

The operational rule:
> **A system whose correctness depends on the model returning the same string twice is
> badly designed. You validate the output; you do not assume it.**

## E. Where it is confidently wrong · 15 min

Three adversarial prompts, already in your grid, each broken in a different way:

- **P4** — count the letter `r` in `strawberry`. The answer is 3.
- **P5** — a question about a fictional company at a date past any training cutoff.
  The only correct answer is that it is not known.
- **P6** — the same counting operation on a French word and an English word, in one
  prompt. The answers are 1 and 3.

Not trick questions: each isolates one consequence of how the thing is built, and each
has a business analogue you will meet inside eighteen months.

In [ ]:

for i in [4,5,6] :
    print("\n" + "=" * 30, f'P{i} Prompt', "=" * 30)
    print(BY_PID[f'P{i}']["prompt"][:500]) # print the first 500 characters of the prompt for P{i}


In [ ]:
adv = grid[grid.family == "adversarial"].copy()

# P4 and P6 have a right answer. P5's only right answer is "I do not know".
EXPECTED = {"P4": ["3"], "P6": ["1, 3", "1,3", "1 and 3"]}
adv["correct"] = [
    bool(row.abstained) if row.prompt_id == "P5"
    else any(normalise(e) in normalise(row.text) for e in EXPECTED[row.prompt_id])
    for row in adv.itertuples()
]

table = adv.pivot_table(index="model", columns="prompt_id", values="correct", aggfunc="mean")
display(table.style.format("{:.0%}"))

hall = 1 - adv.groupby("model").correct.mean()
print("\nHALLUCINATION RATE (share of adversarial calls answered confidently and wrongly):")
for m, v in hall.items():
    print(f"  {m:14s} {v:.0%}")
print("\nFOR YOUR REPORT: this 3x3 table, plus one sentence per prompt on the MECHANISM.")

In [ ]:
# =============================== TODO 4 ======================================
# Your grid used the three prompt families that shipped with it: extraction (extract data), adversarial (prevent hallucination), generative (generate general content). 
#
# For each family, say in one line what a DIFFERENCE BETWEEN MODELS on that family
# would tell you that the other two would not.
#
# A family on which all three models score identically has told you nothing and cost you a third of your budget; a
# family nobody at Velora will ever run has told you something irrelevant.
#
# Then name the family that is MISSING for Velora specifically. There is at least
# one, think about what actually arrives in a support inbox that none of these nine prompts resembles.

FAMILY_JUSTIFICATION = {
    "extraction":  ...,   
    "adversarial": ...,
    "generative":  ...,
}
MISSING_FAMILY = ...      # <- what a Velora grid should have had, and why. Would previous prompt families be relevant to evaluate answers to support mails? Why? 

assert set(FAMILY_JUSTIFICATION) == set(FAMILIES)
assert all(isinstance(v, str) and len(v) > 40 for v in FAMILY_JUSTIFICATION.values())
assert isinstance(MISSING_FAMILY, str) and len(MISSING_FAMILY) > 40
for k, v in FAMILY_JUSTIFICATION.items():
    print(f"{k:12s} {v}")
print(f"\nmissing     {MISSING_FAMILY}")

In [ ]:
# Read the actual answers. The rate is not the finding; the wording is.
for pid in ["P4", "P5", "P6"]:
    print("=" * 74)
    print(f"{pid} — {BY_PID[pid]['prompt'].splitlines()[0][:70]}")
    for name in MODELS:
        got = adv[(adv.model == name) & (adv.prompt_id == pid)].text.iloc[0]
        print(f"  {name:14s} {got[:250]!r}")

**The mechanism, prompt by prompt — this is what goes in your report, not the rate.**

**P4, counting.** The model never sees letters. `strawberry` arrives as two or three
integers, and how many `r` a set of integers contains is a question about a
representation the model does not have. Not a reasoning failure, and no amount of
"think step by step" repairs it — Part F repairs it by changing the representation.

**P5, the confident fabrication.** Nothing in next-token prediction penalises inventing.
A plausible continuation of "Velora's Q1 2026 revenue was" is a number. Abstaining is a
*behaviour added afterwards*, in instruction tuning, by humans who rewarded refusals —
which is why it varies between models and degrades under pressure.

> **Hallucination is the nominal behaviour. What is surprising is how often it is right.**

**P6, two languages.** The same tokenizer cuts French and English words into different
shapes, so the two halves of the prompt are not the same task. A model can be reliable
on one and not the other, and prompt work does not transfer the skill, because the
skill was never about words.

Carry this into next week: **the model does not know your data.** It has never seen
Velora. Session 3 is what it takes to change that, and P5 is why that session exists.

## F. Bonus · whatever time is left

Pick one. A measured negative result is worth more than an unmeasured feature.

- **A fourth model.** Add one to `MODELS` any OpenRouter id and re-run Part C.
  Then check whether the ranking you produced from three survived.
- **Your own language multiplier.** Add a seventh language to `s1_texts.json` — ideally
  one your team actually operates in — and put it in the Part B table.
- **A tenth prompt.** Write one whose failure you can predict *before* you run it, then
  run it. Being wrong about your own prediction is the most instructive outcome
  available here.

In [ ]:
# Scratch space for Part F. Whatever you run, record the number.

## Deliverable

One notebook + one page of prose. Pairs.

Submit:

1. **This notebook, executed** — TODOs filled, outputs visible — **at the end of the
   session**.
2. **A one-page report, by 23:59 on Thursday**, written from the numbers already in this
   notebook:

| | |
|---|---|
| Measurements | the 3 × 3 table: p95 latency, € per 1 000 calls, quality score per model |
| Eye vs measure | your Part A ranking, your Part C ranking, and what the gap between them cost you |
| Tokenisation | your multiplier, and what it does to Velora's bill *and* to the context window |
| Reproducibility | distinct outputs out of 5, and whether the score moved or only the wording |
| Hallucination | the 3 × 3 adversarial table, with a mechanism named per prompt — not just a rate |
| Recommendation | two Velora use cases, two different models, and the metric that flips the decision |
| One limit | one thing your own protocol cannot measure, stated precisely |

**Graded on** (one of six deliverables; your best four count for 30 % of the course):

| | |
|---|---|
| A 3 × 3 table of real measurements | 6 |
| Tokenisation multiplier computed and interpreted | 4 |
| Reproducibility gap measured | 3 |
| Hallucinations characterised, not merely counted | 3 |
| Conditional recommendation: two uses, two models, the criterion that flips it | 3 |
| One explicit limit of your own protocol | 1 |

Prose you could have written before opening this notebook scores zero. 
Every number in your report must name the run it came from and the date it was measured.